# 04 — Controles negativos: fondo/sesión y survivorship

**Proyecto:** `Kaki_DeepHS_StorageDays`

Este notebook responde dos preguntas antes de seleccionar bandas o entrenar modelos más complejos:

1. **¿El fondo de los cubos HSI contiene información suficiente para predecir el día/sesión?**  
   Si el fondo predice `storage_days`, existe una firma temporal/instrumental fuera del tejido del fruto.

2. **¿El espectro basal del día 0 anticipa el día terminal del fruto?**  
   Si ocurre, parte de la aparente predicción temporal puede beneficiarse de diferencias estables entre frutos de maduración rápida y lenta.

Se mantienen los folds externos/internos fruit-disjoint congelados. Este notebook **no hace afirmaciones causales** y no sustituye una campaña independiente.


### Public reproducibility version

This notebook is the cleaned repository version used to reproduce the manuscript analyses. The scientific workflow, frozen fruit-disjoint partitions, model-selection logic, seeds, and evaluation metrics are preserved. Google Drive mounting and chat-specific ZIP export steps were removed. Paths are relative to the repository root. Raw DeepHS Fruit data are not redistributed; see `../docs/DATASET.md`.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    """Locate the cloned repository or use PERSIMMON_HSI_ROOT if defined."""
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Librerías, rutas y configuración


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib, itertools, json, platform, shutil, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from scipy.signal import savgol_filter
from scipy.stats import spearmanr, pearsonr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=RuntimeWarning)

PROJECT='Kaki_DeepHS_StorageDays'
NOTEBOOK='04_kaki_negative_controls_survivorship'
ROOT = PROJECT_ROOT
INTERIM=ROOT/'data/interim'
EXTRACTED=INTERIM/'extracted'
SPLITS=ROOT/'data/splits'
BASE02=ROOT/'results/02_kaki_fruit_disjoint_splits_baselines'
RESULTS=ROOT/'results/04_kaki_negative_controls_survivorship'
FIGURES=ROOT/'figures/04_kaki_negative_controls_survivorship'
TABLES=ROOT/'tables/04_kaki_negative_controls_survivorship'
EXPORT=ROOT/'exports/04_kaki_negative_controls_survivorship'
for p in [RESULTS,FIGURES,TABLES,EXPORT]: p.mkdir(parents=True,exist_ok=True)

SEED=20260818
BOOT=5000
OUTER_FOLDS=5
INNER_FOLDS=4
MIN_BG_PIXELS=80
BG_DILATION=1
EDGE=1
FORCE_BG=False
print(ROOT)


## 2. Cargar espectros aprobados MASK V2, master y splits


In [ ]:
spec_paths={'VIS':INTERIM/'kaki_fruit_day_spectra_VIS.csv',
            'VIS_COR':INTERIM/'kaki_fruit_day_spectra_VIS_COR.csv'}
cube_paths={'VIS':INTERIM/'kaki_cube_spectra_VIS.csv',
            'VIS_COR':INTERIM/'kaki_cube_spectra_VIS_COR.csv'}
expected={'VIS':395,'VIS_COR':398}

spectral_data={}; wavelengths={}; gate=[]
for cam in ['VIS','VIS_COR']:
    c=pd.read_csv(cube_paths[cam])
    med=float(c['mask_coverage'].median())
    gate.append({'camera_type':cam,'n_cubes':len(c),'median_mask_coverage':med,
                 'n_mask_qc_not_ok':int((c['mask_qc_flag']!='ok').sum()),
                 'n_extraction_errors':int(c['extraction_error'].notna().sum())})
    assert len(c)==expected[cam] and med<0.87
    assert (c['mask_qc_flag']=='ok').all() and c['extraction_error'].isna().all()

    d=pd.read_csv(spec_paths[cam])
    d['fruit_id']=pd.to_numeric(d['fruit_id']).astype(int)
    d['storage_days_model']=pd.to_numeric(d['storage_days_model']).astype(int)
    wcols=[x for x in d.columns if x.startswith('wl_')]
    assert len(d)==199 and d['fruit_id'].nunique()==40 and not d[wcols].isna().any().any()
    spectral_data[cam]=d.sort_values(['fruit_id','storage_days_model']).reset_index(drop=True)
    wavelengths[cam]=np.array([float(x.replace('wl_','')) for x in wcols])

gate=pd.DataFrame(gate); display(gate); gate.to_csv(TABLES/'01_mask_v2_gate.csv',index=False)

master=pd.read_csv(INTERIM/'kaki_master_index_v2.csv')
outer=pd.read_csv(SPLITS/'kaki_outer5_fruit_disjoint_v1.csv')
inner=pd.read_csv(SPLITS/'kaki_inner4_within_outer5_fruit_disjoint_v1.csv')
base_oof=pd.read_csv(BASE02/'kaki_oof_predictions.csv')

for d in [master,outer,inner]:
    d['fruit_id']=pd.to_numeric(d['fruit_id']).astype(int)
master['storage_days_model']=pd.to_numeric(master['storage_days_model']).astype(int)
outer['outer_fold']=pd.to_numeric(outer['outer_fold']).astype(int)
inner['outer_fold']=pd.to_numeric(inner['outer_fold']).astype(int)
inner['inner_fold']=pd.to_numeric(inner['inner_fold']).astype(int)
assert len(master)==793 and master['fruit_id'].nunique()==40
print('✅ Datos y splits cargados.')


## 3. Resolver rutas físicas de los 793 cubos


In [ ]:
def resolve(v):
    p=Path(str(v))
    return p if p.is_absolute() else EXTRACTED/p

if 'header_abs' in master:
    master['hdr']=master['header_abs'].map(resolve)
elif 'header_file' in master:
    master['hdr']=master['header_file'].map(resolve)
else:
    raise RuntimeError('Master sin header path.')

if 'data_abs' in master:
    master['bin']=master['data_abs'].map(resolve)
elif 'data_file' in master:
    master['bin']=master['data_file'].map(resolve)
else:
    master['bin']=master['hdr'].map(lambda p:p.with_suffix('.bin'))

assert master['hdr'].map(lambda p:p.exists()).all()
assert master['bin'].map(lambda p:p.exists()).all()
print('✅ 793 pares HDR/BIN resueltos.')


## 4. Lector ENVI y MASK V2


In [ ]:
ENVI_DTYPES={1:np.dtype('u1'),2:np.dtype('i2'),3:np.dtype('i4'),4:np.dtype('f4'),
             5:np.dtype('f8'),12:np.dtype('u2'),13:np.dtype('u4'),14:np.dtype('i8'),15:np.dtype('u8')}

def hdr_meta(path):
    entries={}
    for raw in Path(path).read_text(encoding='utf-8',errors='ignore').splitlines():
        line=raw.strip()
        if not line or line.upper()=='ENVI' or '=' not in line: continue
        k,v=line.split('=',1); entries[k.strip().lower()]=v.strip().strip('{}').strip()
    def I(k,d=None):
        try:return int(float(entries.get(k,d)))
        except:return d
    return {'samples':I('samples'),'lines':I('lines'),'bands':I('bands'),
            'header_offset':I('header offset',0),'data_type':I('data type'),
            'interleave':entries.get('interleave','').lower(),'byte_order':I('byte order',0)}

def open_cube(hdr,binfile,cam):
    m=hdr_meta(hdr); bands=int(m['bands']); wl=wavelengths[cam]
    assert len(wl)==bands
    dt=ENVI_DTYPES[m['data_type']]
    if dt.itemsize>1: dt=dt.newbyteorder('<' if m['byte_order']==0 else '>')
    L,S=int(m['lines']),int(m['samples']); off=int(m['header_offset']); inter=m['interleave']
    if inter=='bip':
        x=np.memmap(binfile,dtype=dt,mode='r',offset=off,shape=(L,S,bands))
    elif inter=='bil':
        x=np.memmap(binfile,dtype=dt,mode='r',offset=off,shape=(L,bands,S)).transpose(0,2,1)
    elif inter=='bsq':
        x=np.memmap(binfile,dtype=dt,mode='r',offset=off,shape=(bands,L,S)).transpose(1,2,0)
    else: raise RuntimeError(f'Interleave no soportado: {inter}')
    return x,wl

def nearest(wl,targets):
    wl=np.asarray(wl)
    return [int(np.argmin(np.abs(wl-t))) for t in targets]

def fruit_mask_v2(cube,wl):
    idx=nearest(wl,(550,700,850))
    proxy=np.nanmedian(np.asarray(cube[:,:,idx],dtype=np.float32),axis=2)
    finite=np.isfinite(proxy); p95=float(np.nanpercentile(proxy[finite],95))
    candidates=[]
    for f in [0.01,0.02,0.03,0.05,0.08,0.10]:
        raw=(proxy>max(1e-4,p95*f)) & finite
        labels,n=ndimage.label(raw)
        if n==0: continue
        sizes=ndimage.sum(raw,labels,index=np.arange(1,n+1))
        mask=labels==(int(np.argmax(sizes))+1)
        mask=ndimage.binary_fill_holes(mask)
        cov=float(mask.mean()); k=6
        corners=np.concatenate([mask[:k,:k].ravel(),mask[:k,-k:].ravel(),
                                mask[-k:,:k].ravel(),mask[-k:,-k:].ravel()])
        cf=float(corners.mean()); score=cov-2*cf
        if cov<0.25:score-=(0.25-cov)*5
        if cov>0.95:score-=(cov-0.95)*10
        candidates.append((score,mask,cov,cf))
    if not candidates: raise RuntimeError('No fruit mask')
    _,mask,cov,cf=max(candidates,key=lambda z:z[0])
    qc='ok' if 0.30<=cov<=0.95 and cf<=0.20 else 'suspect'
    return mask.astype(bool),{'coverage':cov,'corner_fraction':cf,'qc_flag':qc,'proxy':proxy}

def background_mask(mask):
    bg=~ndimage.binary_dilation(mask,iterations=BG_DILATION)
    if EDGE:
        bg[:EDGE,:]=False; bg[-EDGE:,:]=False; bg[:,:EDGE]=False; bg[:,-EDGE:]=False
    method='outside_dilated_fruit'
    if int(bg.sum())<MIN_BG_PIXELS:
        bg=~mask
        if EDGE:
            bg[:EDGE,:]=False; bg[-EDGE:,:]=False; bg[:,:EDGE]=False; bg[:,-EDGE:]=False
        method='fallback_outside_fruit'
    return bg,{'n_background_pixels':int(bg.sum()),
               'background_fraction':float(bg.mean()),'method':method}

r=master.iloc[len(master)//2]
x,wl=open_cube(r['hdr'],r['bin'],r['camera_type'])
m,mm=fruit_mask_v2(x,wl); bg,bm=background_mask(m)
print(r['camera_type'],x.shape,mm['coverage'],bm)


## 5. Diagnóstico estratificado del fondo


In [ ]:
diag=[]; examples=[]
for cam in ['VIS','VIS_COR']:
    dc=master[master['camera_type']==cam]
    for day in sorted(dc['storage_days_model'].unique()):
        g=dc[dc['storage_days_model']==day]
        r=g.iloc[len(g)//2]
        x,wl=open_cube(r['hdr'],r['bin'],cam)
        fm,fmeta=fruit_mask_v2(x,wl); bg,bmeta=background_mask(fm)
        diag.append({'camera_type':cam,'storage_days':int(day),'fruit_id':int(r['fruit_id']),
                     'side':str(r['side']),'fruit_coverage':fmeta['coverage'],
                     'fruit_mask_qc':fmeta['qc_flag'],
                     'n_background_pixels':bmeta['n_background_pixels'],
                     'background_fraction':bmeta['background_fraction'],
                     'background_method':bmeta['method']})
        examples.append((cam,int(day),fmeta['proxy'],fm,bg))
diag=pd.DataFrame(diag); display(diag)
assert (diag['fruit_mask_qc']=='ok').all()
assert (diag['n_background_pixels']>=MIN_BG_PIXELS).all()
diag.to_csv(TABLES/'02_background_mask_diagnostics.csv',index=False)

for cam in ['VIS','VIS_COR']:
    ex=[z for z in examples if z[0]==cam]
    fig,ax=plt.subplots(len(ex),3,figsize=(10,2.3*len(ex)))
    for i,(_,day,proxy,fm,bg) in enumerate(ex):
        ax[i,0].imshow(proxy); ax[i,0].set_title(f'Día {day}: proxy')
        ax[i,1].imshow(fm); ax[i,1].set_title('MASK V2 fruto')
        ax[i,2].imshow(bg); ax[i,2].set_title('Fondo usado')
        for j in range(3): ax[i,j].axis('off')
    fig.suptitle(f'{cam} — diagnóstico fondo')
    fig.tight_layout()
    fig.savefig(FIGURES/f'01_background_mask_diagnostics_{cam}.png',dpi=250,bbox_inches='tight')
    plt.show()
print('✅ Diagnóstico de fondo aprobado.')


## 6. Extraer espectros de fondo de los 793 cubos


In [ ]:
def extract_bg(cam,force=False):
    cache=INTERIM/f'kaki_background_cube_spectra_{cam}_maskV2.csv'
    if cache.exists() and not force:
        print('↪ cache',cache.name); return pd.read_csv(cache)
    sub=master[master['camera_type']==cam]
    rows=[]
    for _,r in tqdm(sub.iterrows(),total=len(sub),desc=f'Background {cam}'):
        out={'camera_type':cam,'fruit_id':int(r['fruit_id']),
             'storage_days_model':int(r['storage_days_model']),
             'day_code':str(r['day_code']),'side':str(r['side']),'extraction_error':None}
        try:
            x,wl=open_cube(r['hdr'],r['bin'],cam)
            fm,fmeta=fruit_mask_v2(x,wl); bg,bmeta=background_mask(fm)
            if bmeta['n_background_pixels']<MIN_BG_PIXELS: raise RuntimeError('Too few background pixels')
            s=np.nanmedian(np.asarray(x[bg,:],dtype=np.float32),axis=0)
            if len(s)!=len(wl) or not np.isfinite(s).all(): raise RuntimeError('Invalid background spectrum')
            out.update({'fruit_mask_coverage':fmeta['coverage'],
                        'n_background_pixels':bmeta['n_background_pixels'],
                        'background_fraction':bmeta['background_fraction'],
                        'background_method':bmeta['method']})
            for w,v in zip(wl,s): out[f'wl_{w:.6f}']=float(v)
        except Exception as e:
            out['extraction_error']=repr(e)
        rows.append(out)
    d=pd.DataFrame(rows); d.to_csv(cache,index=False); return d

bg_cube={cam:extract_bg(cam,FORCE_BG) for cam in ['VIS','VIS_COR']}

qc=[]
for cam,d in bg_cube.items():
    wc=[c for c in d if c.startswith('wl_')]
    qc.append({'camera_type':cam,'n_cubes':len(d),'n_wavelengths':len(wc),
               'n_extraction_errors':int(d['extraction_error'].notna().sum()),
               'bg_pixels_min':int(d['n_background_pixels'].min()),
               'bg_pixels_median':float(d['n_background_pixels'].median()),
               'bg_fraction_median':float(d['background_fraction'].median()),
               'n_nan_spectrum':int(d[wc].isna().any(axis=1).sum())})
bg_qc=pd.DataFrame(qc); display(bg_qc)
assert (bg_qc['n_extraction_errors']==0).all()
assert (bg_qc['n_nan_spectrum']==0).all()
assert (bg_qc['bg_pixels_min']>=MIN_BG_PIXELS).all()
bg_qc.to_csv(TABLES/'03_background_extraction_qc.csv',index=False)


## 7. Agregar front/back del fondo a fruto×día×cámara


In [ ]:
bg_fd={}
for cam,d in bg_cube.items():
    wc=[c for c in d if c.startswith('wl_')]
    s=d.groupby(['fruit_id','storage_days_model','day_code'])[wc].median().reset_index()
    q=d.groupby(['fruit_id','storage_days_model','day_code']).agg(
        n_background_views=('side','size'),
        median_background_pixels=('n_background_pixels','median'),
        median_background_fraction=('background_fraction','median')).reset_index()
    z=s.merge(q,on=['fruit_id','storage_days_model','day_code'])
    z['camera_type']=cam
    z=z.sort_values(['fruit_id','storage_days_model']).reset_index(drop=True)
    assert len(z)==199 and z['fruit_id'].nunique()==40
    bg_fd[cam]=z
    z.to_csv(INTERIM/f'kaki_background_fruit_day_spectra_{cam}_maskV2.csv',index=False)
    print(cam,z['n_background_views'].value_counts().to_dict())


## 8. EDA del fondo: curvas medias, PCA y Spearman


In [ ]:
bg_assoc=[]
for cam,d in bg_fd.items():
    wc=[c for c in d if c.startswith('wl_')]
    wl=np.array([float(c.replace('wl_','')) for c in wc])

    fig,ax=plt.subplots(figsize=(9,5))
    for day,g in d.groupby('storage_days_model'):
        ax.plot(wl,g[wc].mean().to_numpy(float),label=f'Día {int(day)}')
    ax.set(xlabel='Longitud de onda (nm)',ylabel='Señal mediana del fondo',
           title=f'{cam} — fondo medio por día/sesión')
    ax.legend(ncol=2,fontsize=8); ax.grid(alpha=.15); fig.tight_layout()
    fig.savefig(FIGURES/f'02_background_mean_spectra_{cam}.png',dpi=300,bbox_inches='tight'); plt.show()

    X=d[wc].to_numpy(float); Xc=X-X.mean(axis=0,keepdims=True)
    pca=PCA(2); sc=pca.fit_transform(Xc)
    fig,ax=plt.subplots(figsize=(6.4,5.3))
    h=ax.scatter(sc[:,0],sc[:,1],c=d['storage_days_model'],s=38,alpha=.8)
    ax.set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
           ylabel=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)",
           title=f'{cam} — PCA del fondo')
    fig.colorbar(h,ax=ax,label='Storage days'); ax.grid(alpha=.15); fig.tight_layout()
    fig.savefig(FIGURES/f'03_background_pca_{cam}.png',dpi=300,bbox_inches='tight'); plt.show()

    y=d['storage_days_model'].to_numpy(float)
    rho=np.array([spearmanr(d[c].to_numpy(float),y)[0] for c in wc])
    for w,r in zip(wl,rho): bg_assoc.append({'camera_type':cam,'wavelength_nm':float(w),'spearman_rho':float(r)})
    fig,ax=plt.subplots(figsize=(9,4.6)); ax.plot(wl,rho); ax.axhline(0,lw=1)
    ax.set(xlabel='Longitud de onda (nm)',ylabel='Spearman ρ',title=f'{cam} — fondo vs storage day')
    ax.grid(alpha=.15); fig.tight_layout()
    fig.savefig(FIGURES/f'04_background_spearman_{cam}.png',dpi=300,bbox_inches='tight'); plt.show()

bg_assoc=pd.DataFrame(bg_assoc)
bg_assoc.to_csv(TABLES/'04_background_spearman_by_wavelength.csv',index=False)
display(bg_assoc.loc[bg_assoc.groupby('camera_type')['spearman_rho'].apply(lambda s:s.abs().idxmax())])


## 9. Cambio relativo del fruto y del fondo desde día 0


In [ ]:
effects=[]
for cam in ['VIS','VIS_COR']:
    f=spectral_data[cam]; b=bg_fd[cam]
    fw=[c for c in f if c.startswith('wl_')]; bw=[c for c in b if c.startswith('wl_')]
    assert np.allclose([float(c.replace('wl_','')) for c in fw],[float(c.replace('wl_','')) for c in bw],atol=1e-5)
    fm={int(day):g[fw].mean().to_numpy(float) for day,g in f.groupby('storage_days_model')}
    bm={int(day):g[bw].mean().to_numpy(float) for day,g in b.groupby('storage_days_model')}
    for day in sorted(fm):
        fd=fm[day]-fm[0]; bd=bm[day]-bm[0]
        effects.append({'camera_type':cam,'storage_days':day,
            'fruit_relative_l2_change':float(np.linalg.norm(fd)/max(np.linalg.norm(fm[0]),1e-12)),
            'background_relative_l2_change':float(np.linalg.norm(bd)/max(np.linalg.norm(bm[0]),1e-12)),
            'fruit_background_delta_pearson':np.nan if day==0 else float(pearsonr(fd,bd)[0])})
effects=pd.DataFrame(effects); display(effects); effects.to_csv(TABLES/'05_fruit_vs_background_day_effects.csv',index=False)

for cam in ['VIS','VIS_COR']:
    g=effects[effects.camera_type==cam].sort_values('storage_days')
    fig,ax=plt.subplots(figsize=(8,4.6))
    ax.plot(g.storage_days,g.fruit_relative_l2_change,marker='o',label='Fruto')
    ax.plot(g.storage_days,g.background_relative_l2_change,marker='s',label='Fondo')
    ax.set(xlabel='Storage days',ylabel='Cambio L2 relativo desde día 0',title=f'{cam} — deriva fruto vs fondo')
    ax.legend(); ax.grid(alpha=.2); fig.tight_layout()
    fig.savefig(FIGURES/f'05_fruit_vs_background_day_effect_{cam}.png',dpi=300,bbox_inches='tight'); plt.show()


## 10. Funciones de modelado nested Ridge


In [ ]:
PRE=['raw','snv','sg1']; ALPHAS=[.01,.1,1.,10.,100.,1000.]

def prep(X,wl,m):
    X=np.asarray(X,float)
    if m=='raw': return X.copy()
    if m=='snv':
        mu=X.mean(1,keepdims=True); sd=X.std(1,keepdims=True); sd[(~np.isfinite(sd))|(sd==0)]=1
        return (X-mu)/sd
    if m=='sg1':
        return savgol_filter(X,11,2,deriv=1,delta=float(np.median(np.diff(wl))),axis=1,mode='interp')
    raise ValueError(m)

def ridge_pred(Xtr,ytr,Xte,alpha):
    sc=StandardScaler(); A=sc.fit_transform(Xtr); B=sc.transform(Xte)
    m=Ridge(alpha=float(alpha)); m.fit(A,ytr); return np.asarray(m.predict(B),float)

def fb_mae(d):
    z=d.copy(); z['ae']=np.abs(z.y_true-z.y_pred); return float(z.groupby('fruit_id').ae.mean().mean())

def db_mae(d):
    z=d.copy(); z['ae']=np.abs(z.y_true-z.y_pred); return float(z.groupby('y_true').ae.mean().mean())

def metrics(d):
    y=d.y_true.to_numpy(float); p=d.y_pred.to_numpy(float); r=p-y
    rho=spearmanr(y,p)[0]
    return {'n_observations':len(d),'n_fruits':d.fruit_id.nunique(),
            'mae':float(mean_absolute_error(y,p)),
            'rmse':float(np.sqrt(mean_squared_error(y,p))),
            'r2':float(r2_score(y,p)),'spearman_rho':float(rho),
            'within_1_day_accuracy':float(np.mean(np.abs(r)<=1)),
            'fruit_balanced_mae':fb_mae(d),'day_balanced_mae':db_mae(d),
            'bias':float(np.mean(r))}

def inner_select(dev,wl,assign,target='storage_days_model',session_excluded=False):
    wc=[c for c in dev if c.startswith('wl_')]; best=None; rows=[]
    for pp,a in itertools.product(PRE,ALPHAS):
        parts=[]
        for k in range(INNER_FOLDS):
            vf=set(assign.loc[assign.inner_fold==k,'fruit_id'].astype(int))
            tr=dev[~dev.fruit_id.isin(vf)].copy(); va=dev[dev.fruit_id.isin(vf)].copy()
            if not session_excluded:
                Xtr=prep(tr[wc].to_numpy(float),wl,pp); Xv=prep(va[wc].to_numpy(float),wl,pp)
                pred=ridge_pred(Xtr,tr[target].to_numpy(float),Xv,a)
                parts.append(pd.DataFrame({'fruit_id':va.fruit_id.to_numpy(int),'y_true':va[target].to_numpy(float),'y_pred':pred}))
            else:
                for day in sorted(va.storage_days_model.unique()):
                    vd=va[va.storage_days_model==day]; tx=tr[tr.storage_days_model!=day]
                    pred=ridge_pred(prep(tx[wc].to_numpy(float),wl,pp),tx[target].to_numpy(float),
                                    prep(vd[wc].to_numpy(float),wl,pp),a)
                    parts.append(pd.DataFrame({'fruit_id':vd.fruit_id.to_numpy(int),'y_true':vd[target].to_numpy(float),'y_pred':pred}))
        q=pd.concat(parts,ignore_index=True); s1=fb_mae(q); s2=db_mae(q); s3=float(mean_absolute_error(q.y_true,q.y_pred))
        rows.append({'preprocessing':pp,'alpha':a,'inner_fruit_balanced_mae':s1,'inner_day_balanced_mae':s2,'inner_mae':s3})
        rank=(s1,s2,s3,pp,a)
        if best is None or rank<best['rank']: best={'rank':rank,'preprocessing':pp,'alpha':a}
    return best,pd.DataFrame(rows)


## 11. OOF del fondo: Ridge estándar y session-excluded


In [ ]:
all_fruits=set(outer.fruit_id.astype(int))
preds=[]; selected=[]
for cam in ['VIS','VIS_COR']:
    d=bg_fd[cam].copy(); wc=[c for c in d if c.startswith('wl_')]; wl=wavelengths[cam]
    for fold in range(OUTER_FOLDS):
        tf=set(outer.loc[outer.outer_fold==fold,'fruit_id'].astype(int))
        dev=d[~d.fruit_id.isin(tf)].copy(); test=d[d.fruit_id.isin(tf)].copy()
        assign=inner[inner.outer_fold==fold][['fruit_id','inner_fold']].copy()

        # estándar
        best,_=inner_select(dev,wl,assign,session_excluded=False)
        p=ridge_pred(prep(dev[wc].to_numpy(float),wl,best['preprocessing']),
                     dev.storage_days_model.to_numpy(float),
                     prep(test[wc].to_numpy(float),wl,best['preprocessing']),best['alpha'])
        preds.append(pd.DataFrame({'camera_type':cam,'outer_fold':fold,'model':'Background_Ridge_standard',
                                   'fruit_id':test.fruit_id.to_numpy(int),'storage_days':test.storage_days_model.to_numpy(int),
                                   'y_true':test.storage_days_model.to_numpy(float),'y_pred':p}))
        selected.append({'camera_type':cam,'outer_fold':fold,'mode':'standard',**best})

        # session-excluded
        best2,_=inner_select(dev,wl,assign,session_excluded=True)
        for day in sorted(test.storage_days_model.unique()):
            te=test[test.storage_days_model==day]; tr=dev[dev.storage_days_model!=day]
            p=ridge_pred(prep(tr[wc].to_numpy(float),wl,best2['preprocessing']),
                         tr.storage_days_model.to_numpy(float),
                         prep(te[wc].to_numpy(float),wl,best2['preprocessing']),best2['alpha'])
            preds.append(pd.DataFrame({'camera_type':cam,'outer_fold':fold,'model':'Background_Ridge_session_excluded',
                                       'fruit_id':te.fruit_id.to_numpy(int),'storage_days':te.storage_days_model.to_numpy(int),
                                       'y_true':te.storage_days_model.to_numpy(float),'y_pred':p}))
        selected.append({'camera_type':cam,'outer_fold':fold,'mode':'session_excluded',**best2})

bg_model_oof=pd.concat(preds,ignore_index=True)
bg_selected=pd.DataFrame(selected)
for cam in ['VIS','VIS_COR']:
    for mod in ['Background_Ridge_standard','Background_Ridge_session_excluded']:
        g=bg_model_oof[(bg_model_oof.camera_type==cam)&(bg_model_oof.model==mod)]
        assert len(g)==199 and g.fruit_id.nunique()==40
display(bg_selected)
bg_selected.to_csv(TABLES/'06_background_ridge_selected_configs.csv',index=False)


## 12. Baseline de intensidad global del fondo


In [ ]:
parts=[]
for cam in ['VIS','VIS_COR']:
    d=bg_fd[cam].copy(); wc=[c for c in d if c.startswith('wl_')]
    d['global_mean']=d[wc].mean(axis=1)
    for fold in range(OUTER_FOLDS):
        tf=set(outer.loc[outer.outer_fold==fold,'fruit_id'].astype(int))
        tr=d[~d.fruit_id.isin(tf)]; te=d[d.fruit_id.isin(tf)]
        m=LinearRegression().fit(tr[['global_mean']],tr.storage_days_model)
        p=m.predict(te[['global_mean']])
        parts.append(pd.DataFrame({'camera_type':cam,'outer_fold':fold,'model':'Background_GlobalMean',
                                   'fruit_id':te.fruit_id.to_numpy(int),'storage_days':te.storage_days_model.to_numpy(int),
                                   'y_true':te.storage_days_model.to_numpy(float),'y_pred':p}))
bg_global_oof=pd.concat(parts,ignore_index=True)


## 13. Comparación predictiva fruto vs fondo


In [ ]:
fruit_ref=base_oof[base_oof.model=='Ridge'].copy()
fruit_ref['model']='Fruit_Ridge_standard'
control_oof=pd.concat([
    fruit_ref[['camera_type','outer_fold','model','fruit_id','storage_days','y_true','y_pred']],
    bg_model_oof[['camera_type','outer_fold','model','fruit_id','storage_days','y_true','y_pred']],
    bg_global_oof[['camera_type','outer_fold','model','fruit_id','storage_days','y_true','y_pred']]
],ignore_index=True)

rows=[]
for (cam,mod),g in control_oof.groupby(['camera_type','model']):
    z=metrics(g); z.update({'camera_type':cam,'model':mod}); rows.append(z)
control_metrics=pd.DataFrame(rows).sort_values(['camera_type','fruit_balanced_mae']).reset_index(drop=True)
display(control_metrics)
control_metrics.to_csv(TABLES/'07_fruit_vs_background_predictive_metrics.csv',index=False)

order=['Fruit_Ridge_standard','Background_Ridge_standard','Background_GlobalMean','Background_Ridge_session_excluded']
for cam in ['VIS','VIS_COR']:
    g=control_metrics[control_metrics.camera_type==cam].set_index('model').loc[order].reset_index()
    fig,ax=plt.subplots(figsize=(9,4.7)); ax.bar(g.model,g.fruit_balanced_mae)
    ax.set(ylabel='MAE fruit-balanced (días)',title=f'{cam} — fruto vs fondo')
    ax.tick_params(axis='x',rotation=25); ax.grid(axis='y',alpha=.2); fig.tight_layout()
    fig.savefig(FIGURES/f'06_fruit_vs_background_mae_{cam}.png',dpi=300,bbox_inches='tight'); plt.show()


## 14. Bootstrap pareado fondo vs fruto


In [ ]:
def perfruit(d):
    z=d.copy(); z['ae']=np.abs(z.y_true-z.y_pred); return z.groupby('fruit_id').ae.mean()

def paired(a,b,label,n=BOOT,seed=SEED):
    ids=sorted(set(a.index)&set(b.index)); diff=a.loc[ids].to_numpy(float)-b.loc[ids].to_numpy(float)
    rng=np.random.default_rng(seed); boot=np.empty(n)
    for i in range(n):
        idx=rng.integers(0,len(diff),len(diff)); boot[i]=diff[idx].mean()
    return {'comparison':label,'n_fruits':len(diff),'mean_delta_mae_days':float(diff.mean()),
            'ci95_low':float(np.percentile(boot,2.5)),'ci95_high':float(np.percentile(boot,97.5)),
            'fraction_first_better':float(np.mean(diff<0))}

rows=[]
for j,cam in enumerate(['VIS','VIS_COR']):
    bg=control_oof[(control_oof.camera_type==cam)&(control_oof.model=='Background_Ridge_standard')]
    fr=control_oof[(control_oof.camera_type==cam)&(control_oof.model=='Fruit_Ridge_standard')]
    rows.append(paired(perfruit(bg),perfruit(fr),f'{cam}: background minus fruit',seed=SEED+j))
bg_pair=pd.DataFrame(rows); display(bg_pair); bg_pair.to_csv(TABLES/'08_background_vs_fruit_bootstrap.csv',index=False)


## 15. Dataset basal día 0 y target terminal_day


In [ ]:
terminal_map=outer[['fruit_id','terminal_day']].drop_duplicates('fruit_id').set_index('fruit_id').terminal_day.to_dict()
day0={}
for cam,d in spectral_data.items():
    z=d[d.storage_days_model==0].copy()
    assert len(z)==40 and z.fruit_id.nunique()==40
    z['terminal_day']=z.fruit_id.map(terminal_map).astype(int)
    day0[cam]=z.sort_values('fruit_id').reset_index(drop=True)
display(pd.Series(terminal_map).value_counts().sort_index().rename('n_fruits').to_frame())


## 16. PCA y Spearman del día 0 vs terminal_day


In [ ]:
pca_rows=[]; sp_rows=[]
for cam,d in day0.items():
    wc=[c for c in d if c.startswith('wl_')]; wl=wavelengths[cam]; X=d[wc].to_numpy(float); y=d.terminal_day.to_numpy(float)
    for rep in ['raw_centered','snv']:
        Z=X-X.mean(0,keepdims=True) if rep=='raw_centered' else prep(X,wl,'snv')
        Z=Z-Z.mean(0,keepdims=True); p=PCA(2); sc=p.fit_transform(Z)
        pca_rows.append({'camera_type':cam,'representation':rep,
                         'pc1_variance':float(p.explained_variance_ratio_[0]),
                         'pc2_variance':float(p.explained_variance_ratio_[1]),
                         'spearman_terminal_pc1':float(spearmanr(sc[:,0],y)[0]),
                         'spearman_terminal_pc2':float(spearmanr(sc[:,1],y)[0])})
        fig,ax=plt.subplots(figsize=(6.2,5.2)); h=ax.scatter(sc[:,0],sc[:,1],c=y,s=48,alpha=.85)
        ax.set(xlabel=f"PC1 ({p.explained_variance_ratio_[0]*100:.1f}%)",
               ylabel=f"PC2 ({p.explained_variance_ratio_[1]*100:.1f}%)",
               title=f'{cam} — día 0 / {rep}')
        fig.colorbar(h,ax=ax,label='Día terminal'); ax.grid(alpha=.15); fig.tight_layout()
        fig.savefig(FIGURES/f'07_day0_pca_terminal_{cam}_{rep}.png',dpi=300,bbox_inches='tight'); plt.show()
    rho=np.array([spearmanr(d[c].to_numpy(float),y)[0] for c in wc])
    for w,r in zip(wl,rho): sp_rows.append({'camera_type':cam,'wavelength_nm':float(w),'spearman_rho':float(r)})
    fig,ax=plt.subplots(figsize=(9,4.6)); ax.plot(wl,rho); ax.axhline(0,lw=1)
    ax.set(xlabel='Longitud de onda (nm)',ylabel='Spearman ρ',title=f'{cam} — día 0 vs día terminal')
    ax.grid(alpha=.15); fig.tight_layout()
    fig.savefig(FIGURES/f'08_day0_spearman_terminal_{cam}.png',dpi=300,bbox_inches='tight'); plt.show()

day0_pca=pd.DataFrame(pca_rows); day0_sp=pd.DataFrame(sp_rows)
display(day0_pca)
display(day0_sp.loc[day0_sp.groupby('camera_type')['spearman_rho'].apply(lambda s:s.abs().idxmax())])
day0_pca.to_csv(TABLES/'09_day0_pca_summary.csv',index=False)
day0_sp.to_csv(TABLES/'10_day0_spearman_terminal.csv',index=False)


## 17. Nested OOF: predecir terminal_day usando solo día 0


In [ ]:
parts=[]; sel=[]
for cam in ['VIS','VIS_COR']:
    d=day0[cam].copy(); wc=[c for c in d if c.startswith('wl_')]; wl=wavelengths[cam]
    d['global_mean']=d[wc].mean(axis=1)
    for fold in range(OUTER_FOLDS):
        tf=set(outer.loc[outer.outer_fold==fold,'fruit_id'].astype(int))
        dev=d[~d.fruit_id.isin(tf)].copy(); te=d[d.fruit_id.isin(tf)].copy()
        assign=inner[inner.outer_fold==fold][['fruit_id','inner_fold']].copy()
        assert len(dev)==32 and len(te)==8

        # Dummy
        v=float(np.median(dev.terminal_day))
        parts.append(pd.DataFrame({'camera_type':cam,'outer_fold':fold,'model':'Day0_DummyMedian',
                                   'fruit_id':te.fruit_id.to_numpy(int),'y_true':te.terminal_day.to_numpy(float),
                                   'y_pred':np.repeat(v,len(te))}))

        # Global mean
        gm=LinearRegression().fit(dev[['global_mean']],dev.terminal_day)
        p=gm.predict(te[['global_mean']])
        parts.append(pd.DataFrame({'camera_type':cam,'outer_fold':fold,'model':'Day0_GlobalMean',
                                   'fruit_id':te.fruit_id.to_numpy(int),'y_true':te.terminal_day.to_numpy(float),'y_pred':p}))

        # Ridge nested
        best,_=inner_select(dev,wl,assign,target='terminal_day',session_excluded=False)
        p=ridge_pred(prep(dev[wc].to_numpy(float),wl,best['preprocessing']),dev.terminal_day.to_numpy(float),
                     prep(te[wc].to_numpy(float),wl,best['preprocessing']),best['alpha'])
        parts.append(pd.DataFrame({'camera_type':cam,'outer_fold':fold,'model':'Day0_Ridge',
                                   'fruit_id':te.fruit_id.to_numpy(int),'y_true':te.terminal_day.to_numpy(float),'y_pred':p}))
        sel.append({'camera_type':cam,'outer_fold':fold,**best})

day0_oof=pd.concat(parts,ignore_index=True); day0_sel=pd.DataFrame(sel)
rows=[]
for (cam,mod),g in day0_oof.groupby(['camera_type','model']):
    y=g.y_true.to_numpy(float); p=g.y_pred.to_numpy(float); rho=spearmanr(y,p)[0]
    rows.append({'camera_type':cam,'model':mod,'n_fruits':40,
                 'mae':float(mean_absolute_error(y,p)),'rmse':float(np.sqrt(mean_squared_error(y,p))),
                 'r2':float(r2_score(y,p)),'spearman_rho':float(rho),
                 'within_1_day_accuracy':float(np.mean(np.abs(p-y)<=1)),'bias':float(np.mean(p-y))})
day0_metrics=pd.DataFrame(rows).sort_values(['camera_type','mae']).reset_index(drop=True)
display(day0_metrics); display(day0_sel)
day0_metrics.to_csv(TABLES/'11_day0_terminal_predictive_metrics.csv',index=False)
day0_sel.to_csv(TABLES/'12_day0_ridge_selected_configs.csv',index=False)


## 18. Predicho vs real y bootstrap Ridge vs Dummy


In [ ]:
bootrows=[]
for j,cam in enumerate(['VIS','VIS_COR']):
    g=day0_oof[(day0_oof.camera_type==cam)&(day0_oof.model=='Day0_Ridge')]
    fig,ax=plt.subplots(figsize=(6.2,5.3))
    h=ax.scatter(g.y_true,g.y_pred,c=g.fruit_id,s=50,alpha=.85)
    lo=min(-.5,float(g[['y_true','y_pred']].min().min())-.25); hi=max(8.5,float(g[['y_true','y_pred']].max().max())+.25)
    ax.plot([lo,hi],[lo,hi],ls='--'); ax.set(xlim=(lo,hi),ylim=(lo,hi),
        xlabel='Día terminal real',ylabel='Día terminal predicho desde día 0',title=f'{cam} — survivorship control')
    fig.colorbar(h,ax=ax,label='Fruit ID'); ax.grid(alpha=.15); fig.tight_layout()
    fig.savefig(FIGURES/f'09_day0_predicted_terminal_{cam}.png',dpi=300,bbox_inches='tight'); plt.show()

    r=day0_oof[(day0_oof.camera_type==cam)&(day0_oof.model=='Day0_Ridge')].set_index('fruit_id')
    d=day0_oof[(day0_oof.camera_type==cam)&(day0_oof.model=='Day0_DummyMedian')].set_index('fruit_id')
    rae=np.abs(r.y_true-r.y_pred); dae=np.abs(d.y_true-d.y_pred)
    bootrows.append(paired(rae,dae,f'{cam}: Day0 Ridge AE minus Dummy AE',seed=SEED+100+j))
surv_boot=pd.DataFrame(bootrows); display(surv_boot); surv_boot.to_csv(TABLES/'13_day0_ridge_vs_dummy_bootstrap.csv',index=False)


## 19. Tabla integrada de decisión


In [ ]:
decision=[]
for cam in ['VIS','VIS_COR']:
    def cm(mod): return control_metrics[(control_metrics.camera_type==cam)&(control_metrics.model==mod)].iloc[0]
    def dm(mod): return day0_metrics[(day0_metrics.camera_type==cam)&(day0_metrics.model==mod)].iloc[0]
    fr=cm('Fruit_Ridge_standard'); bg=cm('Background_Ridge_standard'); bx=cm('Background_Ridge_session_excluded')
    sr=dm('Day0_Ridge'); sd=dm('Day0_DummyMedian')
    decision.append({'camera_type':cam,
        'fruit_storage_day_mae':float(fr.fruit_balanced_mae),
        'background_storage_day_mae':float(bg.fruit_balanced_mae),
        'background_minus_fruit_mae':float(bg.fruit_balanced_mae-fr.fruit_balanced_mae),
        'background_session_excluded_mae':float(bx.fruit_balanced_mae),
        'background_session_exclusion_delta':float(bx.fruit_balanced_mae-bg.fruit_balanced_mae),
        'day0_terminal_ridge_mae':float(sr.mae),'day0_terminal_dummy_mae':float(sd.mae),
        'day0_ridge_improvement_over_dummy':float(sd.mae-sr.mae),
        'day0_terminal_ridge_r2':float(sr.r2),'day0_terminal_ridge_spearman':float(sr.spearman_rho)})
decision=pd.DataFrame(decision); display(decision); decision.to_csv(TABLES/'14_negative_controls_decision_table.csv',index=False)


## 20. Guardar resultados y resumen JSON


In [ ]:
BG_OOF=RESULTS/'kaki_background_control_oof_predictions.csv'
SURV_OOF=RESULTS/'kaki_day0_survivorship_oof_predictions.csv'
CTRL_MET=RESULTS/'kaki_negative_control_metrics.csv'
bg_all=pd.concat([bg_model_oof,bg_global_oof],ignore_index=True)
bg_all.to_csv(BG_OOF,index=False); day0_oof.to_csv(SURV_OOF,index=False); control_metrics.to_csv(CTRL_MET,index=False)

max_bg=bg_assoc.loc[bg_assoc.groupby('camera_type')['spearman_rho'].apply(lambda s:s.abs().idxmax())].reset_index(drop=True)
max_d0=day0_sp.loc[day0_sp.groupby('camera_type')['spearman_rho'].apply(lambda s:s.abs().idxmax())].reset_index(drop=True)

def js(x):
    if isinstance(x,dict): return {str(k):js(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)): return [js(v) for v in x]
    if isinstance(x,np.integer): return int(x)
    if isinstance(x,np.floating): return None if np.isnan(x) else float(x)
    if isinstance(x,np.bool_): return bool(x)
    if isinstance(x,np.ndarray): return js(x.tolist())
    return x

summary={'project':PROJECT,'notebook':NOTEBOOK,'timestamp':datetime.now().isoformat(timespec='seconds'),
 'background_control':{
    'qc':bg_qc.to_dict('records'),
    'predictive_metrics':control_metrics.to_dict('records'),
    'paired_background_vs_fruit':bg_pair.to_dict('records'),
    'max_abs_band_association':max_bg.to_dict('records'),
    'fruit_vs_background_day_effects':effects.to_dict('records')},
 'survivorship_control':{
    'input':'day-0 fruit spectrum only','target':'terminal_day',
    'metrics':day0_metrics.to_dict('records'),'ridge_vs_dummy_bootstrap':surv_boot.to_dict('records'),
    'max_abs_day0_band_association':max_d0.to_dict('records'),'pca_summary':day0_pca.to_dict('records')},
 'decision_table':decision.to_dict('records'),
 'limitations':[
    'Predictive background is evidence of session/day structure outside fruit tissue, not proof of mechanism.',
    'Non-predictive background would not prove absence of session effects inside fruit pixels.',
    'Predictive day-0 terminal_day indicates stable baseline phenotype/survivorship information, but terminal_day also reflects destructive sampling.',
    'One campaign cannot causally separate storage time from acquisition session.',
    'Independent campaign validation remains necessary for deployment claims.'
 ]}
SUMMARY=RESULTS/'kaki_negative_controls_survivorship_summary.json'
SUMMARY.write_text(json.dumps(js(summary),indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(js(summary),indent=2,ensure_ascii=False))
